# Step 3: Exploratory data analysis

**Input:** `data/raw/city_day.csv`. The Step 2 decisions are re-applied in section 3.0, so this
notebook runs on its own (including in Colab).
**Output:** figures `report/figures/03_*.png`, tables `report/tables/03_*.csv`.

Every section ends with a question the report must answer. The last section turns the findings
into modelling decisions; that list is what links EDA to Steps 4-7.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw" / "city_day.csv"
FIG = ROOT / "report" / "figures"
TAB = ROOT / "report" / "tables"
for d in (FIG, TAB):
    d.mkdir(parents=True, exist_ok=True)

BUCKET_ORDER = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]
UNHEALTHY = ["Poor", "Very Poor", "Severe"]
TRAIN_END = "2019-01-01"          # train = 2015-2018 (Step 4)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 160)

def save(name):
    plt.tight_layout(); plt.savefig(FIG / f"03_{name}.png", dpi=150); plt.show()

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB and not RAW.exists():
    from google.colab import files
    RAW.parent.mkdir(parents=True, exist_ok=True)
    print("Choose city_day.csv from your computer:")
    uploaded = files.upload()
    Path(next(iter(uploaded))).replace(RAW)
print("data file found:", RAW.exists())

## 3.0 Re-apply the Step 2 decisions and define the city sets

Step 2 decisions: negatives to missing (there were none), drop Xylene (61% missing), drop Mumbai
(61% of AQI missing), bucket recomputed from AQI.

**Modelling cities (RQ1, RQ2):** cities with at least 200 non-missing AQI days in the training
period (2015-2018). A city that is absent from training cannot be learned by a model that uses
city as a feature, and its test performance would only measure extrapolation.

In [ ]:
df = pd.read_csv(RAW, parse_dates=["Date"]).sort_values(["City", "Date"]).reset_index(drop=True)
POLLUTANTS = ["PM2.5", "PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3", "Benzene", "Toluene"]
df[POLLUTANTS] = df[POLLUTANTS].mask(df[POLLUTANTS] < 0)
df = df.drop(columns=["Xylene"])
df = df[df["City"] != "Mumbai"].copy()
df["AQI_Bucket"] = pd.cut(df["AQI"], [-np.inf, 50, 100, 200, 300, 400, np.inf],
                          labels=BUCKET_ORDER).astype("object")
df["unhealthy"] = df["AQI_Bucket"].isin(UNHEALTHY).astype(float).where(df["AQI"].notna())

train_aqi_days = df[df["Date"] < TRAIN_END].groupby("City")["AQI"].count()
MODEL_CITIES = sorted(train_aqi_days[train_aqi_days >= 200].index)
df["model_city"] = df["City"].isin(MODEL_CITIES)

city_sets = (df.groupby("City").agg(first=("Date", "min"), aqi_days=("AQI", "count"))
               .join(train_aqi_days.rename("train_aqi_days")).fillna({"train_aqi_days": 0}))
city_sets["model_city"] = city_sets.index.isin(MODEL_CITIES)
city_sets.sort_values("first").to_csv(TAB / "03_city_sets.csv")
print(f"{len(MODEL_CITIES)} modelling cities:", MODEL_CITIES)
print("excluded from RQ1/RQ2:", sorted(set(df["City"]) - set(MODEL_CITIES)))

## 3.1 Summary statistics

In [ ]:
cols = POLLUTANTS + ["AQI"]
stats = df[cols].describe().T[["count", "mean", "50%", "std", "min", "max"]].rename(columns={"50%": "median"})
stats["skew"] = df[cols].skew()
stats["skew_log1p"] = np.log1p(df[cols]).skew()
stats = stats.round(2)
stats.to_csv(TAB / "03_summary_stats.csv")
stats

**For the report:** is mean > median for most pollutants (right skew)? Does `skew_log1p` fall
close to 0? If yes, log1p is justified. Note any pollutant where log1p over-corrects (negative
skew), such as O3: that one may be better left untransformed.

## 3.2 Distributions before and after log1p

In [ ]:
fig, axes = plt.subplots(len(cols), 2, figsize=(10, 2.2 * len(cols)))
for i, c in enumerate(cols):
    x = df[c].dropna()
    sns.histplot(x, bins=60, kde=True, ax=axes[i, 0], color="#4C72B0")
    sns.histplot(np.log1p(x), bins=60, kde=True, ax=axes[i, 1], color="#DD8452")
    axes[i, 0].set_ylabel(c); axes[i, 1].set_ylabel("")
    axes[i, 0].set_xlabel(""); axes[i, 1].set_xlabel("")
axes[0, 0].set_title("raw"); axes[0, 1].set_title("log1p")
save("distributions_raw_vs_log")

## 3.3 Class balance (target for RQ2)

In [ ]:
counts = df["AQI_Bucket"].value_counts().reindex(BUCKET_ORDER)
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=counts.index, y=counts.values, ax=ax, hue=counts.index, legend=False,
            palette=["#2e7d32", "#9ccc65", "#fdd835", "#fb8c00", "#e53935", "#6d1b1b"])
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v}\n{v / counts.sum():.0%}", ha="center", va="bottom", fontsize=9)
ax.set_title("AQI bucket counts (all cities, rows with AQI)"); ax.set_ylabel("days")
save("bucket_counts")

In [ ]:
share = (pd.crosstab(df["City"], df["AQI_Bucket"], normalize="index")
           .reindex(columns=BUCKET_ORDER).fillna(0))
share = share.loc[share[UNHEALTHY].sum(axis=1).sort_values().index]
share.round(3).to_csv(TAB / "03_bucket_share_by_city.csv")
ax = share.plot(kind="barh", stacked=True, figsize=(10, 9), width=0.85,
                color=["#2e7d32", "#9ccc65", "#fdd835", "#fb8c00", "#e53935", "#6d1b1b"])
ax.set_xlabel("share of days"); ax.set_title("AQI bucket mix per city (sorted by share Poor or worse)")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
save("bucket_share_by_city")

In [ ]:
# Positive rate of the binary target per period, modelling cities only.
m = df[df["model_city"]].copy()
m["period"] = np.select([m["Date"] < "2019-01-01", m["Date"] < "2020-01-01"],
                        ["train 2015-18", "val 2019"], "test 2020")
pos = m.groupby("period")["unhealthy"].agg(days="count", positive_rate="mean").round(3)
pos.to_csv(TAB / "03_positive_rate_by_period.csv")
pos

**For the report:** what share of days is Poor or worse? Is it very different between
cities (Delhi vs Bengaluru) and between periods (test 2020 includes the lockdown)? A falling
positive rate from train to test means a model can look worse on test for reasons unrelated
to its quality. This is why ROC-AUC and PR curves are used, not accuracy.

## 3.4 Time patterns

**Pitfall:** cities join the dataset in different years. A pooled yearly average then changes
partly because the *mix of cities* changes, not because air got better or worse. All time
patterns below use the modelling cities only, and the yearly trend uses each city's deviation
from its own mean.

In [ ]:
m["year"] = m["Date"].dt.year; m["month"] = m["Date"].dt.month
pivot = m.pivot_table(index="year", columns="month", values="AQI", aggfunc="mean")
fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(pivot, cmap="YlOrRd", annot=True, fmt=".0f", ax=ax, cbar_kws={"label": "mean AQI"})
ax.set_title("Mean AQI by year and month (modelling cities)")
save("aqi_year_month_heatmap")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for y, g in m.groupby("year"):
    s = g.groupby("month")["AQI"].mean()
    ax.plot(s.index, s.values, marker="o", label=y, lw=2.5 if y == 2020 else 1.2)
ax.set_xticks(range(1, 13)); ax.set_xlabel("month"); ax.set_ylabel("mean AQI")
ax.axvspan(3, 5, color="grey", alpha=0.15, label="lockdown months (2020)")
ax.set_title("Seasonality: monthly mean AQI, one line per year"); ax.legend(ncol=2, fontsize=8)
save("aqi_monthly_by_year")

Caution when reading this plot: these are pooled means, and 2015 contains only about 7 cities,
dominated by Delhi and Ahmedabad. A high 2015 line is partly composition, not dirtier air. The
seasonal *shape* (winter peak, monsoon low) is the reliable message; the city-adjusted trend
below is the right plot for comparing years.

In [ ]:
# Yearly trend, composition-adjusted: AQI minus that city's overall mean.
m["aqi_dev"] = m["AQI"] - m.groupby("City")["AQI"].transform("mean")
trend = m.groupby("year")["aqi_dev"].agg(["mean", "sem", "count"])
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(trend.index, trend["mean"], yerr=1.96 * trend["sem"], marker="o", capsize=4)
ax.axhline(0, color="grey", lw=0.8)
ax.set_ylabel("AQI minus city mean"); ax.set_title("Yearly trend (city-adjusted, 95% CI)")
save("aqi_yearly_trend")
trend.round(1)

In [ ]:
# Day-of-week effect, adjusted for city and month (removes season and city level).
m["aqi_dev_cm"] = m["AQI"] - m.groupby(["City", "year", "month"])["AQI"].transform("mean")
dow = m.groupby(m["Date"].dt.dayofweek)["aqi_dev_cm"].agg(["mean", "sem"])
dow.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(dow.index, dow["mean"], yerr=1.96 * dow["sem"], capsize=4, color="#4C72B0")
ax.axhline(0, color="grey", lw=0.8)
ax.set_ylabel("AQI minus city-month mean"); ax.set_title("Day-of-week effect (95% CI)")
save("aqi_day_of_week")
dow.round(2)

The error bars assume independent days. AQI is strongly autocorrelated (section 3.7), so the
true uncertainty is wider than shown; treat a weekday difference of a few points as weak evidence.

**For the report:** how big is the winter peak relative to the monsoon low? Is the day-of-week
effect large enough to matter, compared with the seasonal swing? (If it is a few AQI points
against a seasonal swing of 100+, day of week will add little to the models.)

### Lockdown as a natural experiment (March to May 2020 vs the same months of 2019)

Same cities, same months, one year apart: this controls for season and city. It does not control
for weather differences between the two years; say so.

In [ ]:
mm = m[m["month"].between(3, 5) & m["year"].isin([2019, 2020])]
lock = mm.groupby(["City", "year"])[["AQI", "PM2.5", "NO2", "CO", "O3"]].mean().unstack("year")
pct = pd.DataFrame({p: (lock[(p, 2020)] / lock[(p, 2019)] - 1) * 100
                    for p in ["AQI", "PM2.5", "NO2", "CO", "O3"]}).dropna(how="all").round(1)
pct.to_csv(TAB / "03_lockdown_pct_change.csv")
fig, ax = plt.subplots(figsize=(9, 4))
pct.median().plot(kind="bar", ax=ax, color="#4C72B0")
ax.axhline(0, color="grey", lw=0.8)
ax.set_ylabel("median % change across cities"); ax.set_title("Mar-May 2020 vs Mar-May 2019")
save("lockdown_change")
pct

**For the report:** which pollutant fell most? NO2 is mostly traffic, so a large NO2 drop is
consistent with less traffic. If O3 did not fall (or rose), that is a known effect: less NO
means less O3 removal. Also note what this implies for Step 4: the 2020 test year is unusual.

## 3.5 City comparison

In [ ]:
order = df.groupby("City")["AQI"].median().sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=df, x="City", y="AQI", order=order, hue="model_city", dodge=False,
            fliersize=1, ax=ax, palette={True: "#4C72B0", False: "#BBBBBB"})
ax.axhline(200, color="#e53935", ls="--", lw=1, label="Poor threshold (200)")
ax.tick_params(axis="x", rotation=60); ax.set_title("AQI per city, ranked by median (grey: not in RQ1/RQ2)")
h, l = ax.get_legend_handles_labels()
ax.legend(h, [{"True": "RQ1/RQ2 city", "False": "not in RQ1/RQ2"}.get(x, x) for x in l], loc="upper right")
save("aqi_by_city")

### Ahmedabad check

Ahmedabad's AQI is far above every other city. The AQI is the maximum of pollutant sub-indices,
so find out which pollutant drives it.

In [ ]:
co = df.groupby("City")["CO"].median().sort_values(ascending=False)
print("median CO (mg/m3) by city, top 5:\n", co.head().round(2))
ahm = df[df["City"] == "Ahmedabad"]
print("\nAhmedabad by year (medians):")
print(ahm.groupby(ahm["Date"].dt.year)[["CO", "PM2.5", "AQI"]].median().round(1))
fig, ax = plt.subplots(figsize=(12, 4))
order_co = co.index
sns.boxplot(data=df, x="City", y="CO", order=order_co, fliersize=1, ax=ax, color="#4C72B0")
ax.set_yscale("log"); ax.tick_params(axis="x", rotation=60)
ax.set_title("CO by city (log scale)")
save("co_by_city")

**For the report:** Ahmedabad's CO is about 10 times any other city, rises from 2015 to 2018 and
collapses in 2020. Under the CPCB scale, CO above 34 mg/m3 alone gives a sub-index above 400
("Severe"), so Ahmedabad's AQI is largely a CO reading. A 10-fold gap with no comparable PM2.5 gap
points to a sensor or calibration problem rather than real air. State the evidence and the
decision taken (see the README / Step 4).

**For the report:** which cities have a median above 200? Grey cities are measured over fewer and
more recent years (often only 2019-2020, including lockdown), so their boxes are not directly
comparable with the others. Say this under the plot.

## 3.6 Relationships between pollutants

Spearman (rank) correlation is used because the variables are heavily skewed; it is unaffected
by the log transform. This is the evidence for using PCR and PCA later.

In [ ]:
corr = df[POLLUTANTS + ["AQI"]].corr(method="spearman")
corr.round(2).to_csv(TAB / "03_spearman_corr.csv")
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(corr, mask=mask, cmap="RdBu_r", vmin=-1, vmax=1, annot=True, fmt=".2f",
            annot_kws={"size": 8}, ax=ax)
ax.set_title("Spearman correlation, pollutants and AQI")
save("corr_heatmap")
pairs = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
             .sort_values(ascending=False))
pairs.head(10).round(2)

In [ ]:
pairs_to_plot = [("PM2.5", "AQI"), ("PM10", "PM2.5"), ("NO", "NOx"),
                 ("NO2", "NOx"), ("NO", "NO2"), ("Benzene", "Toluene")]
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, (x, y) in zip(axes.ravel(), pairs_to_plot):
    d = df[[x, y]].dropna()
    ax.hexbin(np.log1p(d[x]), np.log1p(d[y]), gridsize=50, bins="log", cmap="viridis", mincnt=1)
    rho = d[x].corr(d[y], method="spearman")
    ax.set_xlabel(f"log1p {x}"); ax.set_ylabel(f"log1p {y}"); ax.set_title(f"rho = {rho:.2f}, n = {len(d)}")
save("scatter_pairs")

**For the report:** name the strongly correlated groups (expected: PM2.5 and PM10; NO, NO2 and
NOx; Benzene and Toluene). State explicitly: *correlated predictors make the MLR coefficients
unstable (high VIF), which motivates PCR in RQ1 and PCA in RQ3.* Also comment on the PM2.5 vs
AQI plot: a tight curve with a floor shows AQI is driven by PM2.5 on most days; points above the
curve are days when another pollutant was the binding one.

## 3.7 Autocorrelation: AQI today vs tomorrow

Step 2 confirmed there are no missing dates within a city, so shifting by one row within a city
is the same as shifting by one day.

In [ ]:
a = df.sort_values(["City", "Date"]).copy()
a["AQI_next"] = a.groupby("City")["AQI"].shift(-1)
a["unhealthy_next"] = a.groupby("City")["unhealthy"].shift(-1)
d = a.dropna(subset=["AQI", "AQI_next"])
r = d["AQI"].corr(d["AQI_next"])
fig, ax = plt.subplots(figsize=(6, 6))
ax.hexbin(d["AQI"], d["AQI_next"], gridsize=60, bins="log", cmap="viridis", mincnt=1)
lim = d[["AQI", "AQI_next"]].quantile(0.995).max()
ax.plot([0, lim], [0, lim], color="red", lw=1); ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel("AQI today"); ax.set_ylabel("AQI tomorrow"); ax.set_title(f"Lag-1 correlation r = {r:.3f}")
save("aqi_today_vs_tomorrow")

In [ ]:
# Autocorrelation function, averaged over cities (each city's own series).
lags = range(1, 31)
acf = pd.DataFrame({c: [g["AQI"].autocorr(l) for l in lags]
                    for c, g in a[a["model_city"]].groupby("City")}, index=lags)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(acf.index, acf.median(axis=1), marker="o", label="median over cities")
ax.fill_between(acf.index, acf.min(axis=1), acf.max(axis=1), alpha=0.2, label="min-max")
ax.set_xlabel("lag (days)"); ax.set_ylabel("autocorrelation"); ax.legend()
ax.set_title("AQI autocorrelation by lag (modelling cities)")
save("aqi_acf")

In [ ]:
# How strong is 'tomorrow's label = today's label'? Descriptive only, on train years;
# Step 6 evaluates it properly on validation and test.
t = a[a["model_city"] & (a["Date"] < TRAIN_END)].dropna(subset=["unhealthy", "unhealthy_next"])
tp = ((t["unhealthy"] == 1) & (t["unhealthy_next"] == 1)).sum()
recall = tp / (t["unhealthy_next"] == 1).sum()
precision = tp / (t["unhealthy"] == 1).sum()
print(f"persistence on train years: accuracy {(t['unhealthy'] == t['unhealthy_next']).mean():.3f}, "
      f"precision {precision:.3f}, recall {recall:.3f}")
print(f"P(unhealthy tomorrow | unhealthy today)  = {precision:.3f}")
print(f"P(unhealthy tomorrow | healthy today)    = "
      f"{t.loc[t['unhealthy'] == 0, 'unhealthy_next'].mean():.3f}")

**For the report:** the persistence baseline is strong. The classification models in RQ2 only
add value if they beat it, and the interesting cases are the *transitions* (healthy today,
unhealthy tomorrow), which persistence always misses.

## 3.8 Findings and the modelling decision each one leads to

Fill the "Evidence" column with the numbers from this notebook. Keep this table in the report.

| # | Finding | Evidence | Decision |
|---|---|---|---|
| 1 | Pollutants and AQI are right-skewed | `skew` vs `skew_log1p` | log1p transform (Step 4), except where it over-corrects |
| 2 | Strong correlation within PM, NOx and BTX groups | Spearman table, top pairs | PCR in RQ1, VIF check in MLR, PCA in RQ3 |
| 3 | Target is imbalanced and its rate differs by period | bucket counts, positive rate table | class weights; ROC-AUC, PR-AUC, F1; threshold chosen on validation |
| 4 | Strong winter seasonality | year-month heatmap | cyclic month features (sin, cos) |
| 5 | Day-of-week effect is small or large | day-of-week plot | include day of week or not |
| 6 | 2020 lockdown lowered pollution | lockdown table | report test results also without Mar-May 2020 |
| 7 | Cities differ in level and season | city boxplot, bucket mix | city one-hot in RQ1/RQ2; city profiles in RQ3 |
| 8 | AQI is highly autocorrelated | lag-1 r, ACF | lag features; persistence baseline in RQ2 |
| 9 | Cities enter the data at different times | city sets table | RQ1/RQ2 on modelling cities only; city-adjusted trends |

In [ ]:
if IN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive("step3_outputs", "zip", root_dir=ROOT, base_dir="report")
    files.download("step3_outputs.zip")